# Pipeline kontroli konfoundów — demo

Modułowy pipeline (`confound_pipeline.py`) do wykrywania i neutralizacji konfoundów na embeddingach. Cztery moduły, trzy sposoby definicji konfoundu, usuwanie wielu naraz.

## Moduły
1. **POMIAR** — siła konfoundu (d'), spójność kierunku — czy warto usuwać
2. **DEFINICJA** — detektor CV / etykiety / prompt CLIP
3. **KIERUNEK** — estymacja v_konfound; dla wielu: podprzestrzeń (QR)
4. **NEUTRALIZACJA** — projekcja na dopełnienie ortogonalne + weryfikacja

## Wymaga
Wgraj `confound_pipeline.py` i `detect_hole` (poniżej wbudowany) do Colab. Zdjęcia do `/content`.


## 0. Setup + wgraj bibliotekę

In [ ]:
!pip install -q opencv-python-headless open_clip_torch 2>/dev/null
import numpy as np, cv2
import matplotlib.pyplot as plt
from pathlib import Path
np.set_printoptions(precision=4, suppress=True)

# Wgraj confound_pipeline.py (panel plików Colab lub upload poniżej)
from google.colab import files
import os
if not os.path.exists('confound_pipeline.py'):
    print('Wgraj confound_pipeline.py:')
    files.upload()
from confound_pipeline import ConfoundController, ConfoundDirection
print('✅ Pipeline załadowany')


## 1. Detektor dziurki + naprawa (do metody from_detector)

In [ ]:
def detect_hole(img_bgr):
    h, w = img_bgr.shape[:2]
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    _, dark = cv2.threshold(gray, 30, 255, cv2.THRESH_BINARY_INV)
    m_y, m_x = int(0.12*h), int(0.12*w)
    center = np.zeros_like(dark); center[m_y:h-m_y, m_x:w-m_x] = 1
    dark = dark * center
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5,5))
    dark = cv2.morphologyEx(dark, cv2.MORPH_CLOSE, k)
    contours, _ = cv2.findContours(dark, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    best = None
    for c in contours:
        area = cv2.contourArea(c)
        if area < 80: continue
        perim = cv2.arcLength(c, True)
        if perim == 0: continue
        circ = 4*np.pi*area/(perim*perim)
        (cx,cy), r = cv2.minEnclosingCircle(c)
        fill = area/(np.pi*r*r+1e-6)
        if circ>0.6 and fill>0.55 and 6<r<0.25*min(h,w):
            s = circ*fill
            if best is None or s>best['score']:
                best = dict(cx=cx,cy=cy,radius=r,circularity=circ,fill=fill,score=s)
    return best

def repair_inpaint(img, det):
    mask = np.zeros(img.shape[:2], np.uint8)
    cv2.circle(mask, (int(det['cx']), int(det['cy'])), int(det['radius'])+4, 255, -1)
    return cv2.inpaint(img, mask, 5, cv2.INPAINT_TELEA)

print('Detektor + naprawa gotowe')


## 2. Embedder — CLIP (obraz + tekst) lub fallback

In [ ]:
IMG_EMBED = None
TXT_EMBED = None
try:
    import torch, open_clip
    from PIL import Image
    model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-32', pretrained='laion2b_s34b_b79k')
    tokenizer = open_clip.get_tokenizer('ViT-B-32')
    model.eval()
    def IMG_EMBED(img_bgr):
        pil = Image.fromarray(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
        with torch.no_grad():
            e = model.encode_image(preprocess(pil).unsqueeze(0))
            e = e/e.norm(dim=-1, keepdim=True)
        return e[0].cpu().numpy().astype(np.float64)
    def TXT_EMBED(prompt):
        with torch.no_grad():
            t = model.encode_text(tokenizer([prompt]))
            t = t/t.norm(dim=-1, keepdim=True)
        return t[0].cpu().numpy().astype(np.float64)
    print('✅ CLIP ViT-B/32 (obraz + tekst) — wszystkie 3 metody dostępne')
except Exception as ex:
    print(f'⚠ CLIP niedostępny ({type(ex).__name__}), fallback siatkowy (bez metody clip_prompt)')
    def IMG_EMBED(img_bgr):
        g = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY).astype(np.float32)/255.0
        g = cv2.resize(g,(32,32))
        f = [g[gy*4:(gy+1)*4, gx*4:(gx+1)*4].mean() for gy in range(8) for gx in range(8)]
        e = np.array(f,np.float64); return e/(np.linalg.norm(e)+1e-8)
    TXT_EMBED = None


## 3. Wczytaj zdjęcia (z dziurką i bez)

In [ ]:
IMG_DIR = Path('/content')
exts = ('.jpg','.jpeg','.png')
image_files = sorted([f for f in IMG_DIR.iterdir() if f.suffix.lower() in exts])
images = [cv2.imread(str(f)) for f in image_files]
print(f'Wczytano {len(images)} obrazów')

# Podział na te z dziurką (do detektora) i policz embeddingi wszystkich
has_hole = [detect_hole(im) is not None for im in images]
print(f'  z dziurką: {sum(has_hole)}, bez: {len(images)-sum(has_hole)}')


## 4. Zdefiniuj konfoundy — wszystkie trzy metody

In [ ]:
ctrl = ConfoundController(image_embedder=IMG_EMBED, text_embedder=TXT_EMBED)

# METODA 1: detektor CV (dziurka)
imgs_with_hole = [im for im, h in zip(images, has_hole) if h]
if imgs_with_hole:
    cd_hole = ctrl.from_detector('hole_punch', imgs_with_hole, detect_hole, repair_inpaint)
    print(f'✅ from_detector: {cd_hole}, par={cd_hole.meta["n_pairs"]}')

# METODA 2: etykiety (jasność — dziel obrazy po średniej jasności)
brightness = [cv2.cvtColor(im, cv2.COLOR_BGR2GRAY).mean() for im in images]
median_b = np.median(brightness)
emb_all = np.array([IMG_EMBED(im) for im in images])
dark_emb = emb_all[np.array(brightness) < median_b]
bright_emb = emb_all[np.array(brightness) >= median_b]
if len(dark_emb) and len(bright_emb):
    cd_bright = ctrl.from_labels('brightness', dark_emb, bright_emb)
    print(f'✅ from_labels: {cd_bright}')

# METODA 3: prompt CLIP (jakość ostrości) — tylko jeśli CLIP
if TXT_EMBED is not None:
    cd_sharp = ctrl.from_clip_prompt('sharpness', 'a sharp clear photograph', 'a blurry photograph')
    print(f'✅ from_clip_prompt: {cd_sharp}')
else:
    print('⚠ from_clip_prompt pominięte (brak CLIP text)')

print(f'\nŁącznie zdefiniowano {len(ctrl.directions)} konfoundów')


## 5. POMIAR — które konfoundy są groźne

In [ ]:
# Etykiety zadania: chosen=1, rejected=0. Tu zakładamy że pliki rejected
# mają dziurkę. Dostosuj do swoich danych (np. z nazwy pliku).
labels = np.array([0 if h else 1 for h in has_hole])  # z dziurką -> rejected

print('Siła każdego konfoundu względem etykiety chosen/rejected:\n')
for m in ctrl.measure_all(emb_all, labels):
    cons = f"{m['consistency']:.3f}" if m['consistency'] is not None else '—'
    flag = '⚠ GROŹNY' if m['dprime'] > 1.0 else 'ok'
    print(f"  {m['name']:14} d'={m['dprime']:.3f}  corr={m['corr']:+.3f}  "
          f"spójność={cons}  [{flag}]")

print('\nd\' duże = konfound silnie rozdziela klasy = model by się go nauczył')
print('spójność duża = kierunek powtarzalny = neutralizacja zadziała')


## 6. NEUTRALIZACJA wielu naraz + WERYFIKACJA

In [ ]:
# Usuń WSZYSTKIE konfoundy jednocześnie (podprzestrzeń QR)
emb_clean = ctrl.neutralize(emb_all)
print(f'Embeddingi: {emb_all.shape} -> oczyszczone {emb_clean.shape}\n')

# Pełny raport
ctrl.report(emb_all, labels)


## 7. Wizualizacja — separacja przed i po neutralizacji

In [ ]:
# Pokaż rozkład rzutu na kierunek dziurki przed/po
if 'cd_hole' in dir():
    proj_before = emb_all @ cd_hole.vector
    proj_after = emb_clean @ cd_hole.vector
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    for lab, col, name in [(1,'green','chosen'), (0,'red','rejected')]:
        ax[0].hist(proj_before[labels==lab], bins=15, alpha=0.6, color=col, label=name)
        ax[1].hist(proj_after[labels==lab], bins=15, alpha=0.6, color=col, label=name)
    ax[0].set_title('PRZED neutralizacją\n(rozdzielone = konfound działa)', fontsize=10)
    ax[1].set_title('PO neutralizacji\n(nałożone = konfound usunięty)', fontsize=10)
    for a in ax: a.set_xlabel('rzut na kierunek dziurki'); a.legend()
    plt.tight_layout(); plt.show()
else:
    print('Brak konfoundu hole do wizualizacji')


## 8. Jak użyć w głównym projekcie

```python
# 1. Zbuduj kontroler z embedderem CLIP
ctrl = ConfoundController(image_embedder=clip_img, text_embedder=clip_txt)

# 2. Zdefiniuj konfoundy (dowolna kombinacja metod)
ctrl.from_detector('hole_punch', imgs_killed, detect_hole, repair_inpaint)
ctrl.from_labels('brightness', emb_dark, emb_bright)
ctrl.from_clip_prompt('sharpness', 'sharp photo', 'blurry photo')

# 3. Zmierz (które są groźne?)
ctrl.report(embeddings, labels)

# 4. Oczyść embeddingi PRZED szukaniem osi decydującego momentu
emb_clean = ctrl.neutralize(embeddings)
# ... teraz Akt 1 (decisive_direction) na emb_clean, nie emb
```

### Kluczowa zasada
Neutralizuj konfoundy ZANIM zaczniesz szukać osi decydującego momentu. Inaczej oś może być artefaktem dziurki/jasności zamiast prawdziwym sygnałem.

### Następny krok
Integracja z Aktem 1: liczymy `decisive_direction` na oczyszczonych embeddingach i sprawdzamy, czy separacja chosen/rejected przetrwała neutralizację. Jeśli tak — sygnał jest prawdziwy. Jeśli zniknął — był konfoundem.
